# 05 — Corrective, self-reflective, and active retrieval

**Adaptive RAG Lab · Northstar Insurance policy assistant**

## Scenario

Use execution feedback to recover from poor evidence while enforcing a continuation budget.

Adaptive RAG is a bounded decision system: it selects the smallest retrieval/reasoning policy that can answer a query reliably. In this notebook you will implement and inspect CRAG, Self-RAG, FLARE, evidence graders, bounded loops.

## Theory: adapt the policy, not only the prompt

A fixed RAG pipeline assumes every query should retrieve the same way. An adaptive controller can decide whether retrieval is needed, which source and method fit the information structure, whether to rewrite/decompose, how much evidence to collect, whether the evidence is sufficient, and whether another bounded retrieval cycle is justified.

```text
FLOW (read top to bottom)

+--------------------+
| Query              |
+--------------------+
          |
          v
+--------------------+
| Analyze            |
+--------------------+
          |
          v
+--------------------+
| Retrieve?          |
+--------------------+

Decision branches:
  +-- no --> [Direct answer]
  `-- yes --> [Plan route and budget]

Supporting paths:
  [Plan route and budget] --feeds--> [Retrieve evidence]
  [Retrieve evidence] --feeds--> [Enough, current, authorized?]
  [Enough, current, authorized?] --yes--> [Grounded answer]
  [Enough, current, authorized?] --no + budget--> [Plan route and budget]
  [Enough, current, authorized?] --no budget--> [Abstain / escalate]
```

The goal is not maximum autonomy. It is a measurable quality, cost, latency, and risk trade-off.

In [ ]:
from src.adaptive_rag.router import adaptive_answer, choose_k, classify_query, transform_query

def demo_retrieve(query, k):
    q=query.lower()
    if "policy" in q or "leave" in q:
        return ["current governed policy", "authoritative effective date"][:k]
    if "stale" in q:
        return ["stale policy excerpt"][:k]
    return []

queries = [
    "What is HTTP?",
    "What is our current parental-leave policy?",
    "Compare our 2024 and 2026 parental-leave policies and explain what changed.",
    "Find policy POL-00483.",
    "What are the major themes across all customer interviews?",
]
[(q, classify_query(q)) for q in queries]

## Investigation

Inspect the selected route before judging the answer. A short enterprise question can require retrieval because it is private or current; a long general question may not. The trace should record the selected strategy, rationale, query variants, source/method, K, evidence, evidence-quality judgment, termination reason, latency, and cost.

In [ ]:
for query in queries:
    run = adaptive_answer(query, demo_retrieve)
    print({"query": query, "strategy": run["route"].strategy, "queries": run["queries"], "status": run["status"]})

## Experiment — change one policy decision

1. Change a routing cue and explain the resulting false retrieval or missed retrieval.
2. Compare `k=2`, `k=5`, and `k=10` for a simple and a comparative question.
3. Return stale evidence and confirm that a corrective route reaches abstention once its budget is exhausted.
4. Add an authorization constraint and verify it runs before context construction.

Measure routing accuracy, Recall@K, context precision, answer grounding, latency, cost, tool calls, and recovery length. Compare the adaptive result with a fixed top-K baseline.

In [ ]:
for query in queries:
    decision = classify_query(query)
    print(query[:44], {"route": decision.strategy, "k": choose_k(query, uncertainty=0.6), "rewrites": transform_query(query, decision.strategy)})

## Production checklist and references

- Route by evidence need and risk; do not infer authorization from a prompt.
- Keep routes explicit, budgets finite, and continuation/abstention reasons observable.
- Evaluate strategy selection separately from retrieval and final-answer scores.
- Require a simple-baseline comparison before adding multi-step or agentic behavior.
- Treat all retrieved content as untrusted data.

References: [Adaptive-RAG](https://arxiv.org/abs/2403.14403), [Self-RAG](https://arxiv.org/abs/2310.11511), [CRAG](https://arxiv.org/abs/2401.15884), [FLARE](https://arxiv.org/abs/2305.06983), [Microsoft GraphRAG](https://microsoft.github.io/graphrag/), and the repository’s [Adaptive RAG guide](../../docs/adaptive-rag.md).

**Reflection:** Which additional route would materially improve this scenario? What metric and failure slice would prove it is worth the added complexity?